# 04 — Geographic & Regional Patterns
## Tamil Nadu Assembly Elections (1971–2021)

**Objectives:**
- Regional strongholds heatmap (party × sub_region)
- District-level swing volatility
- Urban vs rural comparison (Chennai City Region vs rest)
- Constituency competitiveness trends (post-2008 delimitation)
- SC/ST constituency analysis

In [ ]:
import sys
sys.path.insert(0, '/app')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from db import query, query_all, GENERAL_ELECTION_YEARS, POST_DELIM_YEARS, MAJOR_PARTIES, SUB_REGIONS

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

df = query_all()
ge = df[df.year.isin(GENERAL_ELECTION_YEARS)].copy()
# Post-delimitation data (234 constituencies, 2011+)
post_delim = ge[ge.year.isin(POST_DELIM_YEARS)].copy()
winners = ge[ge.position == 1].copy()
post_winners = post_delim[post_delim.position == 1].copy()
print(f'Post-delimitation records: {len(post_delim):,}, Winners: {len(post_winners):,}')

## 1. Regional Strongholds Heatmap

In [ ]:
# Party seat share by sub_region for post-delimitation elections
top_parties = ['DMK', 'ADMK', 'INC', 'PMK', 'BJP', 'CPM', 'CPI', 'DMDK', 'MDMK', 'NTK']

region_party = post_winners[post_winners.sub_region.notna()].groupby(['sub_region', 'party']).size().reset_index(name='seats')
region_totals = post_winners[post_winners.sub_region.notna()].groupby('sub_region').size().reset_index(name='total')
region_party = region_party.merge(region_totals, on='sub_region')
region_party['seat_share'] = region_party.seats / region_party.total * 100

# Pivot for heatmap
heatmap_data = region_party[region_party.party.isin(top_parties)].pivot_table(
    index='sub_region', columns='party', values='seat_share', fill_value=0
)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Party Seat Share (%) by Region — Post-Delimitation (2011–2021)', fontsize=14)
ax.set_xlabel('Party')
ax.set_ylabel('Region')
plt.tight_layout()
plt.savefig('/app/output/04_regional_strongholds.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Region-wise party performance per election
for year in POST_DELIM_YEARS:
    yr_winners = post_winners[post_winners.year == year]
    pivot = yr_winners[yr_winners.sub_region.notna() & yr_winners.party.isin(['DMK', 'ADMK', 'INC'])].groupby(
        ['sub_region', 'party']).size().unstack(fill_value=0)
    print(f'\n=== {year} Seats by Region ===')
    print(pivot)

## 2. District-Level Swing Volatility

In [ ]:
# Calculate DMK seat share by district per post-delim election
district_dmk = []
for year in POST_DELIM_YEARS:
    yr = post_winners[(post_winners.year == year) & (post_winners.district_name.notna())]
    for district in yr.district_name.unique():
        d = yr[yr.district_name == district]
        dmk_seats = (d.party == 'DMK').sum()
        total = len(d)
        district_dmk.append({
            'year': year, 'district': district,
            'dmk_share': dmk_seats / total * 100 if total > 0 else 0,
            'total_seats': total
        })

dist_df = pd.DataFrame(district_dmk)

# Swing = std dev of DMK share across elections
swing = dist_df.groupby('district').agg(
    mean_dmk_share=('dmk_share', 'mean'),
    std_dmk_share=('dmk_share', 'std'),
    total_seats=('total_seats', 'first')
).reset_index().sort_values('std_dmk_share', ascending=False)

fig, ax = plt.subplots(figsize=(14, 8))
colors = plt.cm.RdYlGn_r(swing.std_dmk_share / swing.std_dmk_share.max())
ax.barh(swing.district, swing.std_dmk_share, color=colors, edgecolor='white')
ax.set_title('District Swing Volatility (Std Dev of DMK Seat Share, 2011–2021)', fontsize=14)
ax.set_xlabel('Standard Deviation of DMK Seat Share (%)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('/app/output/04_district_swing.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nMost volatile districts (highest swing):')
swing.head(10)

## 3. Urban vs Rural Comparison

In [ ]:
# Compare Chennai City Region vs others
post_delim_with_region = post_delim[post_delim.sub_region.notna()].copy()
post_delim_with_region['is_urban'] = post_delim_with_region.sub_region == 'CHENNAI CITY REGION'
post_delim_with_region['area_type'] = post_delim_with_region.is_urban.map({True: 'Urban (Chennai)', False: 'Rest of TN'})

# Key metrics comparison
comparison = post_delim_with_region.groupby(['year', 'area_type']).agg(
    avg_turnout=('turnout_percentage', 'mean'),
    avg_n_cand=('n_cand', 'mean'),
    avg_enop=('enop', 'mean'),
    avg_margin=('margin', lambda x: x[post_delim_with_region.loc[x.index, 'position'] == 1].mean()),
    female_pct=('sex', lambda x: (x == 'F').mean() * 100)
).reset_index()

metrics = ['avg_turnout', 'avg_n_cand', 'avg_enop', 'female_pct']
titles = ['Avg Turnout (%)', 'Avg Candidates/Constituency', 'Avg ENOP', 'Female Candidate %']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, metric, title in zip(axes.flatten(), metrics, titles):
    for area, color in [('Urban (Chennai)', '#E91E63'), ('Rest of TN', '#2196F3')]:
        d = comparison[comparison.area_type == area]
        ax.plot(d.year, d[metric], 'o-', label=area, color=color, linewidth=2)
    ax.set_title(title)
    ax.set_xlabel('Election Year')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Urban vs Rural Electoral Patterns (2011–2021)', fontsize=14)
plt.tight_layout()
plt.savefig('/app/output/04_urban_rural.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Constituency Competitiveness Trends

In [ ]:
# Margin trends for post-delimitation constituencies
const_margins = post_winners.groupby(['constituency_name', 'year']).agg(
    margin=('margin', 'first'),
    margin_pct=('margin_percentage', 'first'),
    winner_party=('party', 'first')
).reset_index()

# Classify competitiveness
def classify_margin(m):
    if m is None or pd.isna(m): return 'Unknown'
    if m < 5: return 'Ultra-tight (<5%)'
    if m < 10: return 'Competitive (5-10%)'
    if m < 20: return 'Moderate (10-20%)'
    return 'Safe (>20%)'

const_margins['competitiveness'] = const_margins.margin_pct.apply(classify_margin)

comp_by_year = const_margins.groupby(['year', 'competitiveness']).size().unstack(fill_value=0)
comp_order = ['Ultra-tight (<5%)', 'Competitive (5-10%)', 'Moderate (10-20%)', 'Safe (>20%)']
comp_by_year = comp_by_year.reindex(columns=[c for c in comp_order if c in comp_by_year.columns])

fig, ax = plt.subplots(figsize=(14, 6))
comp_by_year.plot(kind='bar', stacked=True, ax=ax, 
                  color=['#E53935', '#FF9800', '#FFC107', '#4CAF50'], edgecolor='white')
ax.set_title('Constituency Competitiveness Distribution (Post-Delimitation)', fontsize=14)
ax.set_xlabel('Election Year')
ax.set_ylabel('Number of Constituencies')
ax.legend(title='Margin Band')
plt.tight_layout()
plt.savefig('/app/output/04_competitiveness.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Track individual constituency competitiveness change
margin_pivot = const_margins.pivot_table(index='constituency_name', columns='year', values='margin_pct')
if 2011 in margin_pivot.columns and 2021 in margin_pivot.columns:
    margin_pivot['change'] = margin_pivot[2021] - margin_pivot[2011]
    margin_pivot = margin_pivot.dropna(subset=['change']).sort_values('change')
    
    fig, ax = plt.subplots(figsize=(14, 10))
    # Show top 15 becoming more competitive and top 15 becoming safer
    top_competitive = margin_pivot.head(15)
    top_safer = margin_pivot.tail(15)
    show = pd.concat([top_competitive, top_safer])
    
    colors = ['#E53935' if c < 0 else '#4CAF50' for c in show.change]
    ax.barh(show.index, show.change, color=colors, edgecolor='white')
    ax.set_title('Constituency Margin Change: 2011 → 2021 (Top 30)', fontsize=14)
    ax.set_xlabel('Change in Winner Margin (%) — Negative = More Competitive')
    ax.axvline(x=0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.savefig('/app/output/04_margin_change.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. SC/ST Constituency Analysis

In [ ]:
# Compare GEN vs SC constituencies
const_type_data = post_delim.copy()
const_type_data = const_type_data[const_type_data.constituency_type.isin(['GEN', 'SC'])]

type_comparison = const_type_data.groupby(['year', 'constituency_type']).agg(
    avg_turnout=('turnout_percentage', 'mean'),
    avg_n_cand=('n_cand', 'mean'),
    avg_margin=('margin_percentage', lambda x: x[const_type_data.loc[x.index, 'position'] == 1].mean()),
    avg_enop=('enop', 'mean'),
    female_pct=('sex', lambda x: (x == 'F').mean() * 100)
).reset_index()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
metrics = ['avg_turnout', 'avg_n_cand', 'avg_margin', 'avg_enop', 'female_pct']
titles = ['Avg Turnout (%)', 'Avg Candidates', 'Avg Winner Margin (%)', 'Avg ENOP', 'Female Candidate %']

for ax, metric, title in zip(axes.flatten(), metrics, titles):
    for ctype, color in [('GEN', '#2196F3'), ('SC', '#FF7043')]:
        d = type_comparison[type_comparison.constituency_type == ctype]
        ax.plot(d.year, d[metric], 'o-', label=ctype, color=color, linewidth=2)
    ax.set_title(title)
    ax.set_xlabel('Year')
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[-1, -1].set_visible(False)
plt.suptitle('General vs SC Reserved Constituencies (2011–2021)', fontsize=14)
plt.tight_layout()
plt.savefig('/app/output/04_sc_vs_gen.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Party preference in SC vs GEN constituencies
sc_gen_party = post_winners[post_winners.constituency_type.isin(['GEN', 'SC'])].groupby(
    ['constituency_type', 'party']).size().reset_index(name='seats')

for ctype in ['GEN', 'SC']:
    print(f'\n=== {ctype} Constituencies — Party Wins (2011–2021) ===')
    d = sc_gen_party[sc_gen_party.constituency_type == ctype].sort_values('seats', ascending=False).head(10)
    for _, row in d.iterrows():
        print(f'  {row.party}: {row.seats} seats')